# VedgeSat driver: New Zealand (default vs NZ classifier)

A copy of the VedgeSat driver for comparing the **default** vegetation classifier (`MLPClassifier_Veg_L5L8S2.pkl`) with the retrained **NZ** classifiers on the same site, dates and settings. No VedgeSat (`Toolshed`) file is edited; the original driver is left as it was.

**How runs are kept apart.** Every run writes to its own folder, `Data/<SITE>_<PERIOD>_<MODEL>`, e.g. `Data/Pakiri_2015-2025_default` and `Data/Pakiri_2015-2025_NZ`. The classifier (`MODEL`) is the only thing that changes between runs of the same `SITE` and `PERIOD`. Cells added or changed for the NZ workflow are marked **(NZ)**.

**Workflow for one site**

1. In <font color='red'>User requirements</font>, set `SITE`, `PERIOD`, the dates and `MODEL = 'default'`. Run from the top to **Save Veglines as Local Shapefiles**, using Option 1 extraction. In `check_detection`, reject an image only for problems with the image itself (cloud, shadow, a bad scene), never because the edge looks worse.
2. Set `MODEL = 'NZ'` and run the same cells again. Nothing else changes.
3. Run **Compare runs (NZ)**. It checks the settings match, lists images kept in one run but not the other, and saves the images common to both.
4. For each model in turn, **default first**: set `MODEL`, run the setup cells down to the reference line load-in, skip Option 1, run **Option 2** load-in, then transects, intersections, plots and validation. Every run uses the default run's transects and only the common images, so the models are compared like for like.

---

**To run a code block, click in a cell, hold down shift, and press enter.** The order in which you execute the code blocks matters. If the heading above a cell is <font color='red'>IN RED</font>, edit that cell with your own requirements and settings. Lines starting with `#` are comments.


### Import Python packages/modules
Load in the packages we'll need for running the tool.

In [1]:
%load_ext autoreload
%autoreload 2
import os
import glob
import pickle
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use('Qt5Agg')
import matplotlib.pyplot as plt
plt.ion()
from datetime import datetime
from Toolshed import Download, Toolbox, VegetationLine, Plotting, PlottingSeaborn, Transects
import ee
import geopandas as gpd
import geemap

ee.Initialize(project='civic-ripsaw-454707-c4')
#ee.Authenticate() # should only need to be run the first time after installation
%gui qt5

### <font color='red'>User requirements</font>
Edit these variables to set up your site of interest, including desired coordinate projections, site name and bounding box, satellite platforms (`L5` is Landsat 5, `L8` is Landsat 8, and `S2` is Sentinel 2) and date range of interest.

In [2]:
# (NZ) Run identity. Each run writes to its own folder: Data/<SITE>_<PERIOD>_<MODEL>
SITE = 'Muriwai'            # site name; also used for the reference line file name below
PERIOD = '2017_winter_testing'        # a label for the date range below, e.g. '2015-2025' (no spaces)
MODEL = 'default'          # classifier for this run: 'default' (VedgeSat's), 'NZ' or 'NZsparse'
sitename = f'{SITE}_{PERIOD}_{MODEL}'

# Date range
dates = ['2017-04-15', '2017-08-03']

# Satellite missions
# Input a list of containing any/all of 'L5', 'L7', 'L8', 'L9', 'S2', 'PSScene4Band'
# L5: 1984-2013; L7: 1999-2017 (SLC error from 2003); L8: 2013-present; S2: 2014-present; L9: 2021-present
#S2 is always preferred for 2015-Present due to higher accuracy (finer spatial/spectral/temporal resolution). If imagery is unavailable, see following note.
sat_list = ['L8', 'S2'] 
#Note: For primary packages where optimal data is available - Using L5 for pre-2013 (excluding 1999-2003 LS7 pre-SLC error), 
# LS8 for 2013-2015, S2 for 2015-Present, L9 for 2021-Present Secondarily. Check specific dates within years for optimal data availability.

# Cloud threshold for screening out cloudy imagery (0.5 or 50% recommended)
cloud_thresh = 0.3 
#0.5 is original - Dependent on clear data availability. A higher threshold will allow more cloudy images but it may not be an issue depending 
# for total swath cloud cover and your particular AOI. A lower threshold will return less images but they will be less cloudy over the entire image.
# Look in the outputs for cloud cover, skip outputs with clear cloud impedence.

# Extract shoreline (wet-dry boundary) as well as veg edge
wetdry = False 
#Set to True or False to add waterline - Note: waterline addition can heavily alter output characteristics/image selection.

# Reference shoreline/veg line shapefile name (should be stored in a folder called referenceLines in Data)
# Line should be ONE CONTINUOUS linestring along the shore, stored as a shapefile in WGS84 coord system - Note: or converted into WGS84
referenceLineShp = 'Muriwai_refLine.shp' # Located in Data/referenceLines
# Maximum amount in metres by which to buffer the reference line for capturing veg edges within.
max_dist_ref = 200 
#150 is original/standard. Adjust based on beach width.

### <font color='red'>Vegetation classifier (NZ)</font>
Chooses the classifier for this run from `MODEL`. VedgeSat picks its classifier inside `Image_Processing.InitialiseImgs`; this cell wraps that function in memory (no file is edited) so that `settings['veg_clf_model']` decides it for Landsat 8 and Sentinel-2. With `MODEL = 'default'` the wrapper changes nothing. Either way it prints the classifier used for each platform once, so you can confirm it during extraction. **Re-run this cell after restarting the kernel.**

In [3]:
VEG_MODELS = {
    'default':  None,                                   # VedgeSat default model: MLPClassifier_Veg_L5L8S2.pkl
    'NZ':       'MLPClassifier_Veg_L8S2_NZ.pkl',        # retrained, sparse_veg not counted as vegetation
    'NZsparse': 'MLPClassifier_Veg_L8S2_NZsparse.pkl',  # retrained, sparse_veg counted as vegetation
}
veg_clf_model = VEG_MODELS[MODEL]

if veg_clf_model is not None:
    model_path = os.path.join(os.getcwd(), 'Classification', 'models', veg_clf_model)
    assert os.path.isfile(model_path), (f'Model not found: {model_path}. Save it with Section 6 of the '
                                        'training notebook, and run this driver from the repo root.')
    other_sats = [s for s in sat_list if s not in ('L8', 'S2')]
    assert not other_sats, (f'The NZ models cover L8 and S2 only; remove {other_sats} from sat_list '
                            '(those images would silently use the default model).')

from Toolshed import Image_Processing
if not hasattr(Image_Processing, '_vedgesat_InitialiseImgs'):      # keep the original, even if this cell is re-run
    Image_Processing._vedgesat_InitialiseImgs = Image_Processing.InitialiseImgs
_reported = set()

def _init_with_chosen_model(metadata, settings, satname, imgs):
    pixel_size, clf_model, ImgColl, init_georef = Image_Processing._vedgesat_InitialiseImgs(metadata, settings, satname, imgs)
    if satname in ['L8', 'S2'] and settings.get('veg_clf_model'):
        clf_model = settings['veg_clf_model']
    if (satname, clf_model) not in _reported:
        print(f'{satname}: vegetation classifier = {clf_model}')
        _reported.add((satname, clf_model))
    return pixel_size, clf_model, ImgColl, init_georef

Image_Processing.InitialiseImgs = _init_with_chosen_model
print(f'Run folder: Data/{sitename}   classifier: {veg_clf_model or "VedgeSat default"}')


Run folder: Data/Muriwai_2017_winter_testing_default   classifier: VedgeSat default


### Set Up Site Directory
Make a series of directories for the new sitename.

In [4]:
# Directory where the data will be stored
filepath = Toolbox.CreateFileStructure(sitename, sat_list)

### Reference Shore Option 1: Define AOI from the bounding box of the <font color='red'>reference shoreline shapefile</font>
You shouldn't need to change anything here; the `referenceLineShp` comes from the <font color='red'>User requirements</font> cell

In [5]:
# Return AOI from reference line bounding box and save AOI folium map HTML in sitename directory
referenceLinePath = os.path.join(filepath, 'referenceLines', referenceLineShp)
referenceLineDF = gpd.read_file(referenceLinePath)

# Added - CRS conversion - for reference lines captured in 2193 or other.
referenceLineDF = referenceLineDF.to_crs(epsg=4326)

# Added - save conversion file back to drive (overwrites old file)
refLine4326 = referenceLinePath.replace('.shp', '_4326.shp')
referenceLineDF.to_file(refLine4326)

polygon, point, lonmin, lonmax, latmin, latmax = Toolbox.AOIfromLine(refLine4326, max_dist_ref, sitename)

# It's recommended to convert the polygon to the smallest rectangle (sides parallel to coordinate axes)       
polygon = Toolbox.smallest_rectangle(polygon)

### Reference Shore Option 2: <font color='red'>Draw desired AOI on map</font>
Use the polygon or rectangle tool in the Leaflet map window (that will appear below the next cell) to draw out your area of interest. Defining only one feature is valid at present. Once you are happy with your AOI, run the next cell to save the feature as your AOI for gathering satellite metadata within.

In [ ]:
# Run this cell to generate an interactive map
Map = geemap.Map(center=[0,0],zoom=2)
Map.add_basemap('HYBRID')
Map

In [ ]:
# Run this cell after defining your region of interest
roi = Map.user_roi.geometries().getInfo()[0]['coordinates']
polygon = [[roi[0][0],roi[0][3],roi[0][1],roi[0][2]]]
point = ee.Geometry.Point(roi[0][0])

polygon, point, BBoxGDF = Toolbox.AOIfromLeaflet(polygon, point, sitename)
NewBBox = BBoxGDF.to_json()


In [ ]:
NewBBox

### Reference Shore Option 3: Define AOI from <font color='red'>user defined coordinates</font> 
Make sure you get the latitude and longitude around the right way, and minimum and maximum correct (including any - signs)

In [ ]:
# EDIT THESE to define an AOI bounding box (minimum and maxiumum latitudes and longitudes)
lonmin, lonmax = -2.84869, -2.79878
latmin, latmax = 56.32641, 56.39814

# Return AOI after checking coords and saving folium map HTML in sitename directory
polygon, point = Toolbox.AOI(lonmin, lonmax, latmin, latmax, sitename)

# It's recommended to convert the polygon to the smallest rectangle (sides parallel to coordinate axes)       
polygon = Toolbox.smallest_rectangle(polygon)

### Compile Input Settings for Gathering Imagery

In [6]:
if len(dates)>2:
    daterange='no'
else:
    daterange='yes'
years = list(Toolbox.daterange(datetime.strptime(dates[0],'%Y-%m-%d'), datetime.strptime(dates[-1],'%Y-%m-%d')))

# Put all the inputs into a dictionary
inputs = {'polygon': polygon, 'dates': dates, 'daterange':daterange, 'sat_list': sat_list, 'sitename': sitename, 'filepath':filepath}

### Image Retrieval
Before downloading the images, check how many images are available for your inputs

In [7]:
inputs = Download.check_images_available(inputs)

Images available between 2017-04-15 and 2017-08-03:
- In Landsat Tier 1 & Sentinel-2 Level-1C:
  L8: 16 images
  S2: 28 images
  Total: 44 images
- In Landsat Tier 2:
  L8: 1 images
  Total: 1 images


### Image Download
Compile the metadata from the Google Earth Engine server

In [8]:
Sat = Download.RetrieveImages(inputs)
# (NZ) CollectMetadata pairs sat_list and Sat by position, and RetrieveImages returns
# Landsat before Sentinel-2. Check they line up before building the metadata.
LANDSAT_PREFIX = {'L5': 'LT05', 'L7': 'LE07', 'L8': 'LC08', 'L9': 'LC09'}
for sat, coll in zip(inputs['sat_list'], Sat):
    if coll.size().getInfo() == 0:
        continue
    first_id = coll.first().get('system:index').getInfo()
    is_landsat = first_id[:4] in LANDSAT_PREFIX.values()
    ok = first_id.startswith(LANDSAT_PREFIX[sat]) if sat in LANDSAT_PREFIX else not is_landsat
    assert ok, (f'sat_list order does not match the retrieved images: "{sat}" was paired with {first_id}. '
                'List Landsat platforms before S2, e.g. sat_list = ["L8", "S2"].')

metadata = Download.CollectMetadata(inputs, Sat)

retrieving image metadata...
making metadata dictionary...
L8 :  100 %   Done
S2 :  100 %   Done


### Local Image Retrieval (Planet)
For the time being, Planet API is very slow at downloading files (depending on the size of your area). You may want to use the [Planet online data](https://developers.planet.com/docs/apps/explorer/) download portal and move the files to local folders that will have been created at the Toolbox.CreateFileStructure() step. 
- Move the image TIFFs into 'Data/YOURSITENAME/local_images/PlanetScope';
- and the respective cloud masks into 'Data/YOURSITENAME/local_images/PlanetScope/cloudmasks'.
- You can move any leftover extra files into 'Data/YOURSITENAME/AuxiliaryImages'

In [9]:
# If you want to include Landsat 7 but DON'T want to include Scan Line Corrector affected images, set SLC=False
Sat = Download.RetrieveImages(inputs, SLC=False)
metadata = Download.CollectMetadata(inputs, Sat)

retrieving image metadata...
Metadata already exists and was loaded


### <font color='red'>Vegetation Edge Settings</font>
ONLY EDIT THESE IF ADJUSTMENTS ARE NEEDED; for example, if you are getting no/disjointed edges, or  if you want to adjust the threshold used for detection in each image.

In [10]:
BasePath = 'Data/' + sitename + '/lines'

if os.path.isdir(BasePath) is False:
    os.mkdir(BasePath)

projection_epsg, _ = Toolbox.FindUTM(polygon[0][0][1],polygon[0][0][0])

settings = {
    # general parameters:
    'cloud_thresh': cloud_thresh,        # threshold on maximum cloud cover
    'output_epsg': projection_epsg,     # epsg code of spatial reference system desired for the output   
    'wetdry': wetdry,              # extract wet-dry boundary as well as veg
    # quality control:
    'check_detection': True,    # if True, shows each shoreline detection to the user for validation
    'adjust_detection': False, # (NZ) keep False for model comparisons: manual threshold changes would mask the classifiers' differences
    'save_figure': True,        # if True, saves a figure showing the mapped shoreline for each image
    # [ONLY FOR ADVANCED USERS] shoreline detection parameters:
    'min_beach_area': 200,     # minimum area (in metres^2) for an object to be labelled as a beach
    'buffer_size': 250,         # radius (in metres) for buffer around sandy pixels considered in the shoreline detection
    'min_length_sl': 500,       # minimum length (in metres) of shoreline perimeter to be valid
    'cloud_mask_issue': False,  # switch this parameter to True if sand pixels are masked (in black) on many images  
    # (NZ) classifier for this run; None = VedgeSat's default. Saved with the run's settings.
    'veg_clf_model': veg_clf_model,
    # add the inputs defined previously
    'inputs': inputs,
    'projection_epsg': projection_epsg,
    'year_list': years,
    # Local thresholding settings (NZ)
    'local_thresh': False,     # refine the threshold in a strip around the first edge
    'local_strip_m': 100,     # strip half-width (m)
}


### <font color='red'>Tidal Information</font>
Compute tides from FES2014 or FES2022 data which is downloaded from the pyFES server. Only relevant for shoreline processing (as it is used to correct for the effects of tides on the cross-shore waterline position). 

(ONLY RUN IF YOU HAVE `pyfes` INSTALLED AND WANT TIDAL INFO SAVED. If you do, change `tidepath` to the path to your `aviso-fes` folder, see the README for details)

Note: FES2022 is more accurate than FES2014 but takes several minutes longer to compute.

In [ ]:
if wetdry is True:
    tidepath = "../aviso-fes/data/fes2014"
    tideoutpath = os.path.join(settings['inputs']['filepath'],'tides',
                            settings['inputs']['sitename']+'_tides_'+
                            settings['inputs']['dates'][0]+'_'+settings['inputs']['dates'][1]+'.csv')
    daterange = dates
    tidelatlon = [(latmin+latmax)/2, (lonmin+lonmax)/2] # centre of bounding box
    Toolbox.ComputeTides(settings,tidepath,tideoutpath,daterange,tidelatlon) 

### Vegetation Edge Reference Line Load-In
Read in a shapefile representing the rough edge of vegetation that you want to investigate along. Does not neet to be accurate as it is used to create a coastal buffer for constraining the edge extraction.

In [11]:
referenceLinePath = os.path.join(inputs['filepath'], 'referenceLines', referenceLineShp)
referenceLine, ref_epsg = Toolbox.ProcessRefline(referenceLinePath,settings)

settings['reference_shoreline'] = referenceLine
settings['ref_epsg'] = ref_epsg
# Distance to buffer reference line by (this is in metres)
settings['max_dist_ref'] = max_dist_ref

### <font color='red'>Reference Image for Coregistration</font>
You can now coregister your satellite images using AROSICS. If you want to try coregistering your images to improve timeseries accuracy, <font color='red'>provide a filepath to a reference RGB image.</font>

In [12]:
settings['reference_coreg_im'] = None # leave as None if no coregistration is to be performed

### Vegetation Line Extraction
**OPTION 1**: Run the extraction tool and return output veg edges as a dictionary of lines with associated image info attached.


In [13]:
output, output_latlon, output_proj = VegetationLine.extract_veglines(metadata, settings, polygon, dates)

Mapping veglines:
Using initial georef: [268252.5, 15, 0, 5925047.5, 0, -15]
L8: vegetation classifier = MLPClassifier_Veg_L5L8S2.pkl
L8:   6.250 %  - Skipped: cloud threshold exceeded (53.8%)
L8:   12.500 %  - Skipped: cloud threshold exceeded (40.0%)
L8:   18.750 %  
saving LANDSAT/LC08/C02/T1_TOA/LC08_073086_20170524
 
saving classified LANDSAT/LC08/C02/T1_TOA/LC08_073086_20170524
 
saving transition zone of LANDSAT/LC08/C02/T1_TOA/LC08_073086_20170524
L8:   25.000 %  - Skipped: cloud threshold exceeded (37.8%)
L8:   31.250 %  - Skipped: empty/low quality raster
L8:   37.500 %  
saving LANDSAT/LC08/C02/T1_TOA/LC08_073086_20170711
 
saving classified LANDSAT/LC08/C02/T1_TOA/LC08_073086_20170711
 
saving transition zone of LANDSAT/LC08/C02/T1_TOA/LC08_073086_20170711
L8:   43.750 %  - Skipped: cloud threshold exceeded (73.6%)
L8:   50.000 %  - Skipped: cloud threshold exceeded (59.3%)
L8:   56.250 %  - Skipped: cloud threshold exceeded (58.1%)
L8:   62.500 %  
saving LANDSAT/LC08/C02/

### Vegetation Line Extraction Load-In
**OPTION 2**: Load in pre-existing output file generated from a previous run of the vegetation edge extraction tool.


In [ ]:
output, output_latlon, output_proj = Toolbox.ReadOutput(inputs)

# Remove Duplicate Lines
# For images taken on the same date by the same satellite, keep only the longest line
output = Toolbox.RemoveDuplicates(output) 

### Save Veglines as Local Shapefiles
You **DO NOT** need to run this again if you have already exported a previous run to a shapefile, especially not if you have then edited/tidied any unwanted lines for further analysis in an external GIS software (as rerunning this cell will overwrite your edits!).

In [145]:
# Save output veglines 
Toolbox.SaveConvShapefiles(output, BasePath, sitename, settings['output_epsg'])
# Save output shorelines if they were generated
if settings['wetdry'] == True:
    Toolbox.SaveConvShapefiles_Water(output, BasePath, sitename, settings['output_epsg'])

saving shapefile to Data/SouthTaranakiBight_2022_Validation_vedgesat_default/lines\SouthTaranakiBight_2022_Validation_vedgesat_default_2022-02-27_2022-04-27_veglines.shp


### Compare runs (NZ)
Run once the extraction and shapefile save are done for every model in `COMPARE`, with the same `SITE` and `PERIOD`. It:

1. checks the runs' settings are identical apart from the classifier and the folder name;
2. lists images with an edge in one run but not another (rejected in `check_detection`, or no edge found);
3. saves the images common to all runs to `Data/<SITE>_<PERIOD>_common_images.csv`. The transect cell below then keeps only those images in every run.

If an image is in one run only because you rejected it there for a problem with the image, the common list drops it from the other run too, which is what you want.

In [ ]:
COMPARE = ['default', 'NZ']     # models run for this SITE and PERIOD (add 'NZsparse' if you ran it)
runs = {m: f'{SITE}_{PERIOD}_{m}' for m in COMPARE}

# 1. Settings: only the classifier (and the folder name) should differ
def load_settings(name):
    path = os.path.join(filepath, name, name + '_settings.pkl')
    assert os.path.isfile(path), f'No settings found for {name}: has its extraction been run? ({path})'
    with open(path, 'rb') as f:
        return pickle.load(f)

def comparable(s):
    s = dict(s)
    s['inputs'] = {k: v for k, v in s.get('inputs', {}).items() if k != 'sitename'}
    s.pop('veg_clf_model', None)
    return s

base = comparable(load_settings(runs[COMPARE[0]]))
for m in COMPARE[1:]:
    other = comparable(load_settings(runs[m]))
    diffs = [k for k in sorted(set(base) | set(other)) if str(base.get(k)) != str(other.get(k))]
    if diffs:
        print(f'WARNING: {COMPARE[0]} and {m} runs differ in: {diffs}')
        for k in diffs:
            print(f'  {k}:\n    {COMPARE[0]}: {base.get(k)}\n    {m}: {other.get(k)}')
    else:
        print(f'OK: {COMPARE[0]} and {m} runs have identical settings apart from the classifier.')

# 2. Images with an edge in each run
def images(name):
    out, _, _ = Toolbox.ReadOutput(dict(inputs, sitename=name))
    dates = [str(d)[:10] for d in out['dates']]
    sats = out.get('satname', [''] * len(dates))
    return set(zip(dates, sats))

kept = {m: images(runs[m]) for m in COMPARE}
common = set.intersection(*kept.values())
print(f'\nImages with an edge: ' + ', '.join(f'{m} {len(v)}' for m, v in kept.items()) + f';  in all runs: {len(common)}')
for m, v in kept.items():
    only = sorted(v - common)
    if only:
        print(f'  not in every run, will be dropped from {m}: {only}')

# 3. Save the common images for the transect step
common_path = os.path.join(filepath, f'{SITE}_{PERIOD}_common_images.csv')
pd.DataFrame(sorted(common), columns=['dates', 'satname']).to_csv(common_path, index=False)
print(f'\nSaved {len(common)} common images to {common_path}')


### <font color='red'>Define Settings for Cross-shore Transects</font>
Edit me to define the characteristics of the cross-shore transects generated for intersecting with the extracted vegetation edges.

In [ ]:
# Note to self: Learn about these - look more into cross-shore transects
SmoothingWindowSize = 21 
NoSmooths = 100
TransectSpacing = 10
DistanceInland = 100
DistanceOffshore = 100

# Provide average beach slope (tanBeta) for site, for calculating corrected beach widths
# Set to 'None' if you want to use CoastSat.slope to calculate per-transect slopes for correcting with
beachslope = None

### Create Cross-shore Transects
Generate transects based on the above settings.

In [ ]:
VegBasePath = 'Data/' + sitename + '/lines'
VeglineShp = glob.glob(BasePath+'/*veglines.shp')
VeglineGDF = gpd.read_file(VeglineShp[0])
VeglineGDF = VeglineGDF.sort_values(by='dates') # sort GDF by dates to ensure transect intersects occur in chronological order
VeglineGDF = VeglineGDF.reset_index(drop=True) # reset GDF index after date sorting

# (NZ) Keep only images with an edge in every compared run (saved by Compare runs)
common_path = os.path.join(filepath, f'{SITE}_{PERIOD}_common_images.csv')
if os.path.isfile(common_path):
    common = pd.read_csv(common_path, dtype=str).fillna('')
    line_dates = VeglineGDF['dates'].astype(str).str[:10]
    if 'satname' in VeglineGDF.columns and common['satname'].str.len().gt(0).any():
        keys = set(zip(common['dates'], common['satname']))
        keep = [k in keys for k in zip(line_dates, VeglineGDF['satname'].astype(str))]
    else:
        keep = line_dates.isin(set(common['dates'])).tolist()
    print(f'Common images: keeping {sum(keep)} of {len(VeglineGDF)} veglines in {sitename}')
    VeglineGDF = VeglineGDF[keep].reset_index(drop=True)
else:
    print('No common-images file yet: using all veglines (run Compare runs first for a model comparison)')
if settings['wetdry'] == True:
    WaterlineShp = glob.glob(BasePath+'/*waterlines.shp')
    WaterlineGDF = gpd.read_file(WaterlineShp[0])
    WaterlineGDF = WaterlineGDF.sort_values(by='dates') # as above with VeglineGDF date sorting
    WaterlineGDF = WaterlineGDF.reset_index(drop=True)
# Produces Transects for the reference line
TransectSpec =  os.path.join(BasePath, sitename+'_Transects.shp')

# (NZ) Every model uses the default run's transects, so edges are measured along identical lines
default_name = f'{SITE}_{PERIOD}_default'
default_T = os.path.join('Data', default_name, 'lines', default_name + '_Transects.pkl')

# If transects already exist, load them in
if os.path.isfile(TransectSpec[:-3]+'pkl'):
    print('Transects already exist and were loaded')
    with open(TransectSpec[:-3]+'pkl', 'rb') as Tfile: 
        TransectGDF = pickle.load(Tfile)
elif MODEL != 'default' and os.path.isfile(default_T):
    print(f'Using the transects from {default_name}')
    with open(default_T, 'rb') as Tfile:
        TransectGDF = pickle.load(Tfile)
else:
    if MODEL != 'default':
        print('WARNING: no transects from the default run yet; making new ones. Run transects for the default model first.')
    TransectGDF = Transects.ProduceTransects(settings, SmoothingWindowSize, NoSmooths, TransectSpacing, DistanceInland, DistanceOffshore, VegBasePath, referenceLineShp)
    
# make new transect intersections folder
if os.path.isdir(os.path.join(filepath, sitename, 'intersections')) is False:
    os.mkdir(os.path.join(filepath, sitename, 'intersections'))

### Transect-Veg Intersections
Create (or load existing) a GeoDataFrame holding intersections with veg edges along each transect.

**(NZ)** If you ran this before saving the common images, delete `Data/<sitename>/intersections/<sitename>_transect_intersects.pkl` first, or the cached intersections (with all images) are loaded instead.


In [ ]:
if os.path.isfile(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_intersects.pkl')):
    print('Transect Intersect GDF exists and was loaded')
    with open(os.path.join
              (filepath , sitename, 'intersections', sitename + '_transect_intersects.pkl'), 'rb') as f:
        TransectInterGDF = pickle.load(f)
else:
    # Get intersections
    TransectInterGDF = Transects.GetIntersections(BasePath, TransectGDF, VeglineGDF)
    # Save newly intersected transects as shapefile
    TransectInterGDF = Transects.SaveIntersections(TransectInterGDF, VeglineGDF, BasePath, sitename)
    # Repopulate dict with intersection distances along transects normalised to transect midpoints
    TransectInterGDF = Transects.CalculateChanges(TransectInterGDF)
    
    with open(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_intersects.pkl'), 'wb') as f:
        pickle.dump(TransectInterGDF, f)

### Transect-Water Intersections
Create (or load existing) a GeoDataFrame holding intersections with waterlines along each transect. 

In [ ]:
if os.path.isfile(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_water_intersects.pkl')):
    print('Transect Intersect + Water GDF exists and was loaded')
    with open(os.path.join
              (filepath , sitename, 'intersections', sitename + '_transect_water_intersects.pkl'), 'rb') as f:
        TransectInterGDFWater = pickle.load(f)
else:        
    if settings['wetdry'] == True:
        TransectInterGDFWater = Transects.GetWaterIntersections(BasePath, TransectGDF, TransectInterGDF, WaterlineGDF, settings, output)  
    
    with open(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_water_intersects.pkl'), 'wb') as f:
        pickle.dump(TransectInterGDFWater, f)


### Transect-Waves Intersections
Create (or load existing) a GeoDataFrame holding intersections with topographic data along each transect. This is for comparing veg edge positions with nearshore wave conditions at the time the image was taken. NOTE: this requires you to have a Copernicus Marine Service (CMEMS) account with access to their hindcast model, as you will be asked for a username and password. This should also be run before the tidal corrections and beach with steps if you want to include runup in your corrections.

In [ ]:

if os.path.isfile(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_wave_intersects.pkl')):
    print('Transect Intersect + Wave GDF exists and was loaded')
    with open(os.path.join
              (filepath , sitename, 'intersections', sitename + '_transect_wave_intersects.pkl'), 'rb') as f:
        TransectInterGDFWave = pickle.load(f)
else:
    TransectInterGDFWave = Transects.WavesIntersect(settings, TransectInterGDF, BasePath, output, lonmin, lonmax, latmin, latmax)
    
    with open(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_wave_intersects.pkl'), 'wb') as f:
        pickle.dump(TransectInterGDFWave, f)

### Additional wave-based WL metrics
This is for comparing shoreline change with vegetation change, and for quantifying the beach width between the two for each image. If you would like to include runup in your waterline corrections, add `TransectInterGDFWave` to `GetWaterIntersections()`:

```TransectInterGDFWater = Transects.GetWaterIntersections(BasePath, TransectGDF, TransectInterGDF, WaterlineGDF, settings, output, TransectInterGDFWave, beachslope)```

If you want to include runup AND calculate slopes using CoastSat.slope (recommended), exclude the `beachslope` variable:

`TransectInterGDFWater = Transects.GetWaterIntersections(BasePath, TransectGDF, TransectInterGDF, WaterlineGDF, settings, output, TransectInterGDFWave)`

In [ ]:

if 'wlcorrdist' not in TransectInterGDFWater.columns:
    # Tidal correction to get corrected distances along transects
    TransectInterGDFWater = Transects.WLCorrections(settings, output, TransectInterGDFWater, TransectInterGDFWave)     
    # Calculate width between VE and corrected WL
    TransectInterGDFWater = Transects.CalcBeachWidth(settings, TransectGDF, TransectInterGDFWater)
    # Calculate rates of change on corrected WL and save as Transects shapefile
    TransectInterGDFWater = Transects.SaveWaterIntersections(TransectInterGDFWater, WaterlineGDF,  BasePath, sitename)
    with open(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_water_intersects.pkl'), 'wb') as f:
        pickle.dump(TransectInterGDFWater, f)

### Transect-Topo Intersections
Create (or load existing) a GeoDataFrame holding intersections with topographic data along each transect. This is for comparing veg edge positions with dune slopes. <font color='red'>Edit the TIF filename</font> if you want to run this intersection process.

In [ ]:
# EDIT ME: Path to slope raster for extracting slope values
TIF = '/path/to/Slope_Raster.tif'

if os.path.isfile(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_topo_intersects.pkl')):
    print('Transect Intersect + Topo GDF exists and was loaded')
    with open(os.path.join
              (filepath , sitename, 'intersections', sitename + '_transect_topo_intersects.pkl'), 'rb') as f:
        TransectInterGDFTopo = pickle.load(f)
else:
    # Update Transects with Transition Zone widths and slope if available
    TransectInterGDFTopo = Transects.TZIntersect(settings, TransectInterGDF, VeglineGDF, BasePath)
    TransectInterGDFTopo = Transects.SlopeIntersect(settings, TransectInterGDFTopo, VeglineGDF, BasePath, TIF)
    
    with open(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_topo_intersects.pkl'), 'wb') as f:
        pickle.dump(TransectInterGDFTopo, f)

### <font color='red'>Plotting</font>
Edit the desired Transect IDs to plot timeseries of veg change and beach width change across 

In [ ]:
# Timeseries Plotting

# EDIT ME: Select transect ID to plot
# You can plot subplots within a list of plot IDs, e.g. [[sub1, sub2], plot2]
# You can also comment Line 1 out and uncomment Line 2 to create plots for ALL Transect IDs
# NOTE: If you want to plot ALL transects, it's recommended you switch ShowPlot=False

TransectIDs = [[25,30,35],50,75] # Line 1
# TransectIDs = list(TransectInterGDF['TransectID']) # Line 2

for TransectID in TransectIDs:
    # Plot timeseries of cross-shore veg position
    Plotting.VegTimeseries(sitename, TransectInterGDF, TransectID, Hemisphere='S', ShowPlot=True)
    # If plotting veg and water lines together
    if settings['wetdry']:
        Plotting.VegWaterTimeseries(sitename, TransectInterGDFWater, TransectID, Hemisphere='S', ShowPlot=True)


In [ ]:
# Beach Width Plotting

# Select transect ID to plot
TransectIDs = [[25,30,35],50,75]
for TransectID in TransectIDs:
    # Plot timeseries of cross-shore width between water edge and veg edge 
    Plotting.WidthTimeseries(sitename, TransectInterGDFWater, TransectID, Hemisphere='S')


## OPTIONAL: Validation
### <font color='red'>Validation Settings</font>
Most likely you won't need to validate your lines, but if you do, edit these parameters.

In [ ]:
# Name of date column in validation edges shapefile (case sensitive!) 
DatesCol = 'Date'
# (NZ) EDIT ME: your manually digitised vegetation edges for this site
ValidationShp = f'./Validation/{SITE}_manual_veglines.shp'
# EDIT ME: List of transect ID tuples (startID, finishID); default is every transect
TransectIDList = [(0, len(TransectGDF) - 1)]

In [ ]:
# Satellite Edges Validation
validpath = os.path.join(os.getcwd(), 'Data', sitename, 'validation')

if os.path.isfile(os.path.join(validpath, sitename + '_valid_intersects.pkl')):
    print('ValidDict exists and was loaded')
    with open(os.path.join(validpath, sitename + '_valid_intersects.pkl'), 'rb') as f:
        ValidInterGDF = pickle.load(f)
else:
    ValidInterGDF = Transects.ValidateSatIntersects(sitename, ValidationShp, DatesCol, TransectGDF, TransectInterGDF)
    with open(os.path.join(validpath, sitename + '_valid_intersects.pkl'), 'wb') as f:
        pickle.dump(ValidInterGDF, f)


# Quantify errors between validation and satellite derived lines
for TransectIDs in TransectIDList:
    Toolbox.QuantifyErrors(sitename, VeglineGDF,'dates',ValidInterGDF,TransectIDs)

### <font color='red'>Validation Plots</font>
Plot violin plots, probability density functions and regression lines of the cross-shore distance along each transect between each satellite-derived veg edge and validation veg edge.

In [ ]:
# EDIT ME: List of transect ID tuples (startID, finishID)
TransectIDList = [(0, len(TransectGDF) - 1)]

for TransectIDs in TransectIDList:
    PlotTitle = 'Accuracy of Transects ' + str(TransectIDs[0]) + ' to ' + str(TransectIDs[1])
    PlottingSeaborn.SatViolin(sitename,VeglineGDF,'dates',ValidInterGDF,TransectIDs, PlotTitle)
    PlottingSeaborn.SatPDF(sitename,VeglineGDF,'dates',ValidInterGDF,TransectIDs, PlotTitle)
    Plotting.SatRegress(sitename,VeglineGDF,'dates',ValidInterGDF,TransectIDs, PlotTitle)